# N-real · 真实 RAG: 检索 (gpt2 嵌入) + 接地生成 (TinyLlama)

> **小而真** (配套 rag-essential) · RAG = 先检索相关文档, 再让模型基于文档回答。
> 这里**全程真实**: 用 gpt2 隐状态当嵌入做检索, 把检索到的文档喂给 TinyLlama 生成。
> 看 RAG 怎么把「闭卷瞎编」变成「开卷有据」。CPU 离线确定性。

In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1] / "_shared"))
import realmodels as rm
import numpy as np, torch
print("真实模型可用性:", rm.available())

C:\Users\ericp\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


真实模型可用性: {'gpt2': True, 'TinyLlama/TinyLlama-1.1B-Chat-v1.0': True}


> 若上面显示某模型为 False, 表示本机无该 HF 缓存, 真实例子会自动跳过 (不影响本专题的 toy notebook)。

## 1. 一个模型不可能知道的「冷知识」语料库

In [2]:
CORPUS = [
    "The Zorblax Prize 2024 was awarded to Dr. Mira Chen for her work on tidal desalination.",
    "Photosynthesis converts light energy into chemical energy stored in glucose.",
    "The Heaviside layer reflects radio waves and helps long-distance transmission.",
    "Mount Kilimanjaro is the highest mountain in Africa, located in Tanzania.",
]
QUESTION = "Who won the Zorblax Prize in 2024?"
GOLD = "Dr. Mira Chen"   # 只有 CORPUS[0] 含答案; 模型预训练里没有 (虚构事实)
print("问题:", QUESTION)
print("黄金答案只在文档0里:", CORPUS[0])

问题: Who won the Zorblax Prize in 2024?
黄金答案只在文档0里: The Zorblax Prize 2024 was awarded to Dr. Mira Chen for her work on tidal desalination.


## 2. 用真实 gpt2 嵌入做检索 (隐状态均值池化 + 余弦相似)

In [3]:
import matplotlib, matplotlib.pyplot as plt
matplotlib.rcParams['axes.unicode_minus']=False
for f in ['Microsoft YaHei','SimHei','DejaVu Sans']:
    try: matplotlib.rcParams['font.sans-serif']=[f]; break
    except Exception: pass
tok, gmodel = rm.gpt2()
def embed(text):
    ids = tok(text, return_tensors='pt')
    with torch.no_grad():
        hs = gmodel(**ids, output_hidden_states=True).hidden_states[-1][0]
    return hs.mean(0).numpy()                    # 均值池化 = 句向量

if gmodel is not None:
    docvecs = np.stack([embed(d) for d in CORPUS])
    qv = embed(QUESTION)
    sims = docvecs @ qv / (np.linalg.norm(docvecs,axis=1)*np.linalg.norm(qv)+1e-8)
    rank = np.argsort(-sims)
    print("检索相似度排序:")
    for r in rank:
        print(f"  doc{r}  sim={sims[r]:+.3f}  {CORPUS[r][:55]}...")
    top_doc = CORPUS[rank[0]]
    print(f"\n→ 检索到的 top 文档: doc{rank[0]} ({'命中✅' if rank[0]==0 else '未命中⚠'})")
else:
    print("无 gpt2, 跳过"); top_doc = CORPUS[0]

检索相似度排序:
  doc0  sim=+0.999  The Zorblax Prize 2024 was awarded to Dr. Mira Chen for...
  doc3  sim=+0.998  Mount Kilimanjaro is the highest mountain in Africa, lo...
  doc2  sim=+0.998  The Heaviside layer reflects radio waves and helps long...
  doc1  sim=+0.996  Photosynthesis converts light energy into chemical ener...

→ 检索到的 top 文档: doc0 (命中✅)


## 3. 闭卷 vs 开卷 (接地): 同一问题, 给不给检索文档

In [4]:
tok2, lm = rm.tinyllama()
if lm is not None:
    closed = rm.chat(tok2, lm, QUESTION + " Answer in one short sentence.", max_new_tokens=40)
    grounded = rm.chat(tok2, lm,
        f"Context: {top_doc}\n\nQuestion: {QUESTION}\nAnswer using only the context, in one short sentence.",
        max_new_tokens=40)
    print("【闭卷 (无检索)】", closed.replace(chr(10),' ')[:180])
    print("  命中黄金答案?", GOLD.lower() in closed.lower())
    print()
    print("【开卷 (RAG 接地)】", grounded.replace(chr(10),' ')[:180])
    print("  命中黄金答案?", GOLD.lower() in grounded.lower())
else:
    print("无 TinyLlama, 跳过")

【闭卷 (无检索)】 The Zorblax Prize was awarded to the team that developed the most innovative and effective solution to the challenges posed by the Zorblax, a new type of energy source that harness
  命中黄金答案? False

【开卷 (RAG 接地)】 Dr. Mira Chen won the Zorblax Prize in 2024 for her work on tidal desalination.
  命中黄金答案? True


## 4. 反思
你跑了一条**全真实**的 RAG 流水线: gpt2 嵌入检索 → TinyLlama 接地生成。带走:
- **闭卷**: 模型对预训练里没有的事实只能瞎编 (幻觉)。
- **开卷 (RAG)**: 把相关文档塞进上下文, 模型「照着读」就能答对。
- RAG 两段都可能出错: **检索错** (召回不到/排错序) 或 **生成不忠实** (不照文档答)。
  这对应你 rag-essential 学的两大评估面: 检索质量 + 生成忠实度 (groundedness)。

> RAG 的价值: 把「模型记住多少」变成「模型会查多少」, 知识可更新、可溯源。检索质量是上限。